In [4]:
import pandas as pd

df = pd.read_excel("data/rapido_data.xlsx")
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               100000 non-null  int64         
 1   tpep_pickup_datetime   100000 non-null  datetime64[us]
 2   tpep_dropoff_datetime  100000 non-null  datetime64[us]
 3   passenger_count        100000 non-null  int64         
 4   trip_distance          100000 non-null  float64       
 5   pickup_longitude       100000 non-null  float64       
 6   pickup_latitude        100000 non-null  float64       
 7   RatecodeID             100000 non-null  int64         
 8   store_and_fwd_flag     100000 non-null  str           
 9   dropoff_longitude      100000 non-null  float64       
 10  dropoff_latitude       100000 non-null  float64       
 11  payment_type           100000 non-null  int64         
 12  fare_amount            100000 non-null  float64       
 

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,1,2016-03-01,2016-03-01 00:07:55,1,2.50,-73.976746,40.765152,1,N,-74.004265,40.746128,1,9.0,0.5,0.5,2.05,0.00,0.3,12.35
1,1,2016-03-01,2016-03-01 00:11:06,1,2.90,-73.983482,40.767925,1,N,-74.005943,40.733166,1,11.0,0.5,0.5,3.05,0.00,0.3,15.35
2,2,2016-03-01,2016-03-01 00:31:06,2,19.98,-73.782021,40.644810,1,N,-73.974541,40.675770,1,54.5,0.5,0.5,8.00,0.00,0.3,63.80
3,2,2016-03-01,2016-03-01 00:00:00,3,10.78,-73.863419,40.769814,1,N,-73.969650,40.757767,1,31.5,0.0,0.5,3.78,5.54,0.3,41.62
4,2,2016-03-01,2016-03-01 00:00:00,5,30.43,-73.971741,40.792183,3,N,-74.177170,40.695053,1,98.0,0.0,0.0,0.00,15.50,0.3,113.80


In [5]:
# 1. Deduplicate at dataset level and assign clean surrogate primary key (trip_id)
df = df.drop_duplicates().reset_index(drop=True)
df["trip_id"] = df.index

# 2. Ensure datetime types
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# 3. Build datetime_dim
datetime_dim = df[["tpep_pickup_datetime", "tpep_dropoff_datetime"]].reset_index(drop=True)
datetime_dim["pick_hour"] = datetime_dim["tpep_pickup_datetime"].dt.hour
datetime_dim["pick_day"] = datetime_dim["tpep_pickup_datetime"].dt.day
datetime_dim["pick_month"] = datetime_dim["tpep_pickup_datetime"].dt.month
datetime_dim["pick_year"] = datetime_dim["tpep_pickup_datetime"].dt.year
datetime_dim["pick_weekday"] = datetime_dim["tpep_pickup_datetime"].dt.weekday

datetime_dim["drop_hour"] = datetime_dim["tpep_dropoff_datetime"].dt.hour
datetime_dim["drop_day"] = datetime_dim["tpep_dropoff_datetime"].dt.day
datetime_dim["drop_month"] = datetime_dim["tpep_dropoff_datetime"].dt.month
datetime_dim["drop_year"] = datetime_dim["tpep_dropoff_datetime"].dt.year
datetime_dim["drop_weekday"] = datetime_dim["tpep_dropoff_datetime"].dt.weekday

datetime_dim["datetime_id"] = datetime_dim.index
datetime_dim = datetime_dim[
    [
        "datetime_id",
        "tpep_pickup_datetime",
        "pick_hour",
        "pick_day",
        "pick_month",
        "pick_year",
        "pick_weekday",
        "tpep_dropoff_datetime",
        "drop_hour",
        "drop_day",
        "drop_month",
        "drop_year",
        "drop_weekday",
    ]
]

# 4. Build passenger_count_dim & trip_distance_dim
passenger_count_dim = df[["passenger_count"]].reset_index(drop=True)
passenger_count_dim["passenger_count_id"] = passenger_count_dim.index
passenger_count_dim = passenger_count_dim[["passenger_count_id", "passenger_count"]]

trip_distance_dim = df[["trip_distance"]].reset_index(drop=True)
trip_distance_dim["trip_distance_id"] = trip_distance_dim.index
trip_distance_dim = trip_distance_dim[["trip_distance_id", "trip_distance"]]

# 5. Build ride_category_dim (mapped from RatecodeID for Rapido)
ride_category_map = {
    1: "Rapido Bike",
    2: "Rapido Auto",
    3: "Rapido Cab Economy",
    4: "Rapido Cab Premium",
    5: "Rapido Parcel",
    6: "Outstation",
}
rate_code_dim = df[["RatecodeID"]].reset_index(drop=True)
rate_code_dim["rate_code_id"] = rate_code_dim.index
rate_code_dim["rate_code_name"] = rate_code_dim["RatecodeID"].map(ride_category_map)
rate_code_dim = rate_code_dim[["rate_code_id", "RatecodeID", "rate_code_name"]]

# 6. Build pickup_location_dim & dropoff_location_dim
pickup_location_dim = df[["pickup_longitude", "pickup_latitude"]].reset_index(drop=True)
pickup_location_dim["pickup_location_id"] = pickup_location_dim.index
pickup_location_dim = pickup_location_dim[
    ["pickup_location_id", "pickup_latitude", "pickup_longitude"]
]

dropoff_location_dim = df[["dropoff_longitude", "dropoff_latitude"]].reset_index(drop=True)
dropoff_location_dim["dropoff_location_id"] = dropoff_location_dim.index
dropoff_location_dim = dropoff_location_dim[
    ["dropoff_location_id", "dropoff_latitude", "dropoff_longitude"]
]

# 7. Build payment_type_dim (mapped for Rapido payment modes)
payment_type_map = {
    1: "UPI",
    2: "Cash",
    3: "Rapido Wallet",
    4: "Credit/Debit Card",
    5: "Unknown",
    6: "Voided Trip",
}
payment_type_dim = df[["payment_type"]].reset_index(drop=True)
payment_type_dim["payment_type_id"] = payment_type_dim.index
payment_type_dim["payment_type_name"] = payment_type_dim["payment_type"].map(payment_type_map)
payment_type_dim = payment_type_dim[["payment_type_id", "payment_type", "payment_type_name"]]

# 8. Build central fact_table joining all surrogate keys
fact_table = (
    df.merge(passenger_count_dim, left_on="trip_id", right_on="passenger_count_id")
    .merge(trip_distance_dim, left_on="trip_id", right_on="trip_distance_id")
    .merge(rate_code_dim, left_on="trip_id", right_on="rate_code_id")
    .merge(pickup_location_dim, left_on="trip_id", right_on="pickup_location_id")
    .merge(dropoff_location_dim, left_on="trip_id", right_on="dropoff_location_id")
    .merge(datetime_dim, left_on="trip_id", right_on="datetime_id")
    .merge(payment_type_dim, left_on="trip_id", right_on="payment_type_id")[
        [
            "trip_id",
            "VendorID",
            "datetime_id",
            "passenger_count_id",
            "trip_distance_id",
            "rate_code_id",
            "store_and_fwd_flag",
            "pickup_location_id",
            "dropoff_location_id",
            "payment_type_id",
            "fare_amount",
            "extra",
            "mta_tax",
            "tip_amount",
            "tolls_amount",
            "improvement_surcharge",
            "total_amount",
        ]
    ]
)

print("Fact table shape:", fact_table.shape)
fact_table.head() 

Fact table shape: (100000, 17)


C:\Users\jaggu\AppData\Local\Temp\ipykernel_18920\1908465084.py:15: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  datetime_dim["pick_weekday"] = datetime_dim["tpep_pickup_datetime"].dt.weekday
C:\Users\jaggu\AppData\Local\Temp\ipykernel_18920\1908465084.py:21: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  datetime_dim["drop_weekday"] = datetime_dim["tpep_dropoff_datetime"].dt.weekday


,trip_id,VendorID,datetime_id,passenger_count_id,trip_distance_id,rate_code_id,store_and_fwd_flag,pickup_location_id,dropoff_location_id,payment_type_id,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount
0,0,1,0,0,0,0,N,0,0,0,9.0,0.5,0.5,2.05,0.00,0.3,12.35
1,1,1,1,1,1,1,N,1,1,1,11.0,0.5,0.5,3.05,0.00,0.3,15.35
2,2,2,2,2,2,2,N,2,2,2,54.5,0.5,0.5,8.00,0.00,0.3,63.80
3,3,2,3,3,3,3,N,3,3,3,31.5,0.0,0.5,3.78,5.54,0.3,41.62
4,4,2,4,4,4,4,N,4,4,4,98.0,0.0,0.0,0.00,15.50,0.3,113.80
